# 09 — Interpretabilidade com SHAP

Utiliza SHAP no conjunto de validação de 2024 para medir impacto e direção das features nas previsões. O conjunto de 2025 permanece bloqueado.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
PASTA_MODELOS = os.path.join(PASTA_PROJETO, "outputs", "modelos")
PASTA_INTERPRETACAO = os.path.join(PASTA_PROJETO, "outputs", "interpretabilidade")
os.makedirs(PASTA_INTERPRETACAO, exist_ok=True)

df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet"))
df["br_categoria"] = df["br"].astype("Int64").astype("string").fillna("Não informado")

pipeline = joblib.load(os.path.join(PASTA_MODELOS, "xgboost.joblib"))
preprocessador = pipeline.named_steps["preprocessamento"]
modelo = pipeline.named_steps["modelo"]

In [ ]:
colunas_tracado = [c for c in df.columns if c.startswith("tracado_") and c != "tracado_via"]
categoricas = ["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas = ["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"] + colunas_tracado
features = categoricas + numericas

validacao = df[df["ano"] == 2024].copy()
amostra = validacao.sample(n=min(5000, len(validacao)), random_state=42)
X_amostra = amostra[features]

X_transformado = preprocessador.transform(X_amostra)
nomes_features = preprocessador.get_feature_names_out()
X_shap = X_transformado.toarray() if hasattr(X_transformado, "toarray") else X_transformado
print("Amostra SHAP:", X_shap.shape)

In [ ]:
explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list):
    shap_values = shap_values[-1]

media_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = (pd.DataFrame({"feature": nomes_features, "mean_abs_shap": media_abs_shap})
                   .sort_values("mean_abs_shap", ascending=False)
                   .reset_index(drop=True))
display(shap_importance.head(30))
shap_importance.to_csv(
    os.path.join(PASTA_INTERPRETACAO, "shap_importance.csv"),
    index=False, encoding="utf-8-sig"
)

In [ ]:
shap.summary_plot(
    shap_values, X_shap, feature_names=nomes_features,
    max_display=25, show=False
)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_INTERPRETACAO, "shap_summary_plot.png"),
            dpi=300, bbox_inches="tight")
plt.show()

shap.summary_plot(
    shap_values, X_shap, feature_names=nomes_features,
    plot_type="bar", max_display=25, show=False
)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_INTERPRETACAO, "shap_bar_plot.png"),
            dpi=300, bbox_inches="tight")
plt.show()

print("SHAP concluído. 2025 permanece reservado.")